# 03 - Crop-type label QC (thematic validation)

Demonstrates `fieldqc.run_qc`, the interactive tkinter tool for reviewing crop-type labels against field photos.
The tool was used for thematic validation for data publication.
It is intended to also enable detailed QA and labeling to match your specific use case.

Output of a session is four QC columns written into the input GeoDataFrame:

| column | values |
|---|---|
| `qc_reviewed` | `True` for every visited row, `None` otherwise (used for resume). |
| `qc_flag` | `"correct"` · `"mismatch"` · `"skip"` · `None`. |
| `qc_crop_main` | corrected main crop label, `None` if unchanged. |
| `qc_crop_sec` | corrected secondary crop, `"__removed__"` if removed, `None` if unchanged. |

From this output a summary is printed upon session completion.

## Keybindings

```
A   Accept / correct          (both labels confirmed correct)
D   Mismatch -> sub-menu:
      W   Relabel main crop
      E   Relabel secondary crop
      F   Relabel both
      Q   Cancel
E   Skip   (image unusable OR too ambiguous to assess)
Q   Go back one step
R   End session (quit)
```

## Flag semantics

- **correct** -- reviewer confirmed both labels.
- **mismatch** -- reviewer identified at least one wrong label and corrected it.
- **skip** -- image cannot be assessed (too dark, not a field, post harvest, corrupted, or too ambiguous). *Not* a label error: excluded from the accuracy denominator and reported separately.

Only `"correct"` and `"mismatch"` rows enter the accuracy calculations in notebook `04_summary.ipynb`.

In [ ]:
from pathlib import Path

import geopandas as gpd

from fieldqc import run_qc

## Inputs
We start from the `data/` directory containing the published files including the `DCIM` folder per campaign. The QC tool reads the main crop label from `crop_main` and the secondary from `crop_sec`.



In [ ]:
DATA_DIR = Path("../data")
CAMPAIGN = "01_LRS23"
CODE     = "LRS23"

gpkg_path       = DATA_DIR / CAMPAIGN / f"CropHype-Fields-Kenya_{CODE}.gpkg"
image_dir       = DATA_DIR / CAMPAIGN / "DCIM"
checkpoint_path = DATA_DIR / CAMPAIGN / f"{CODE}_qc_checkpoint.gpkg"    # will be generated automatically to track progress across sessions
output_path     = DATA_DIR / CAMPAIGN / f"CropHype-Fields-Kenya_{CODE}_QC.gpkg"

gdf = gpd.read_file(gpkg_path)
print(f"{CODE}: {len(gdf)} fields, {int(gdf['photo'].notna().sum())} with photos")

## Launch field-QC tool

The defaults below draw a stratified random sample of 500 fields, balanced across `crop_main`, with a fixed seed for reproducibility. Set `n_samples=None` to review every field with a photo on disk.

If `checkpoint_path` exists from a previous session, progress is restored automatically and a resume dialog appears.

NOTE: known handling peculiarity: if backstepping [Q] onto a field that was previously marked mismatch and corrected, the tool overwrites this correction unless it is entered again.

In [ ]:
run_qc(
    gdf,
    image_col="photo",
    crop_col="crop_main",
    sec_crop_col="crop_sec",
    image_dir=image_dir,
    id_col="field_id",
    checkpoint_path=checkpoint_path,
    n_samples=500,
    random_seed=42,
    stratify_col="crop_main",
)